# AI Reviewer, Advanced Showcase

Each showcase is a small group of cells:
- **Setup** cell (imports plus inputs or a real model)
- **Computation** cell (the code the reviewer analyzes and refines), and a
- `%perfmonitor_ai_review` call that hands that computation to the reviewer. 

Every example links back to the original repository notebook.

### Dependencies


In [ ]:
%pip install -q -e ".."
# Optional: enable LLM suggestions in %perfmonitor_ai_review (also needs a provider API key):
# %pip install -q -e "..[ai]"

# Fetch demo model weights / sample images into resources/ (kept out of git; safe to re-run)
!python fetch_resources.py

## Setup

Load the extension

>Note that: The AI review itself needs the AI extra packages and an API key: 
> - You need first install: `pip install jumper-extension[ai]` 
> - And set your model provider key.


In [1]:
%load_ext jumper_extension

[JUmPER]: ADLXPybind not available. AMD GPU monitoring disabled.
[JUmPER]: Perfmonitor extension loaded


Run the quick setup (starts performance monitoring and turns on automatic
perf reports)

In [2]:
%perfmonitor_fast_setup

[JUmPER]: Enabled ipympl interactive plots
[JUmPER] Compiling native_c monitor binary (first use, one-time step)...
[JUmPER] Building C collector: /usr/bin/make -C /home/ub/projects/work/jumper-ws/jumper_ipython_extension/jumper_extension/monitor/backends/native_c jumper_collector
[JUmPER] C collector compiled successfully.
[JUmPER] native_c monitor compiled successfully.
[JUmPER] Using native_c monitor (compiled C collector).
[JUmPER]: Performance monitoring started (PID: 38568, Interval: 1.0s)
[JUmPER]: Performance monitoring already running
[JUmPER]: Performance reports enabled for each cell (level: process, interval: 1.0, format: html)
[JUmPER]: Fast setup complete! Ready for interactive analysis.


## A. CPU-bound loops, NumPy vectorization

Pure-Python numeric loops that should collapse into vectorized array ops. Expected reviewer tag: `cpu_bound`.

*All examples in this section come from the **PythonDataScienceHandbook** repo (jakevdp).*


### A1 · Element-wise reciprocals via a Python loop
**Issue:** explicit index loop computing `1/x` element by element, instead of a vectorized `1.0 / values`.
🔗 [`02.03-Computation-on-arrays-ufuncs.ipynb`](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/02.03-Computation-on-arrays-ufuncs.ipynb) · cells 3 & 5

Let's see what the Jumper AI reviewer makes of this one.


In [3]:
import numpy as np

rng = np.random.default_rng(seed=1701)
big_array = rng.integers(1, 100, size=2_500_000)   # original idiom, scaled up

[JUmPER]: No performance data available or recorded cells are too short


In [4]:
def compute_reciprocals(values):
    output = np.empty(len(values))
    for i in range(len(values)):
        output[i] = 1.0 / values[i]
    return output

result = compute_reciprocals(big_array)

Metric,AVG,MIN,MAX,Total/Limit
CPU Util (Across 12 CPUs),8.92,8.92,8.92,-
Memory (GB),0.26,0.26,0.27,15.58
GPU Util (Across 1 GPUs),0.00,0.00,0.00,-
GPU Memory (GB),0.00,0.00,0.00,12.00


In [5]:
%perfmonitor_ai_review --strategy deep --benchmark

[JUmPER]: benchmarking 3 suggestion(s) against cell 2; each replays 2 preceding cell(s) 3 time(s). This can take a while.


In [6]:
%perfmonitor_ai_review --resume 7e5a8568 --select 1

[JUmPER]: No pending AI review found for run_id '7e5a8568'


In [ ]:
def compute_reciprocals(values):
    return 1.0 / values

result = compute_reciprocals(big_array)

### A2 · Nested arithmetic accumulation loop
**Issue:** millions of scalar iterations in a double loop that is fully vectorizable with `np.arange` plus broadcasting.
🔗 [`01.07-Timing-and-Profiling.ipynb`](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/01.07-Timing-and-Profiling.ipynb) · cells 5 & 12


In [3]:
total = 0
for i in range(4000):
    for j in range(4000):
        total += i * (-1) ** j

memory_limit = 15.58
gpu_memory_limit = 12.0
ratios: {'memory': np.float64(0.014812986735130509), 'cpu': np.float64(0.08527719333333335), 'gpu_util': 0.0, 'gpu_memory': 0.0}
ranked_tags = []


Metric,AVG,MIN,MAX,Total/Limit
CPU Util (Across 12 CPUs),8.53,8.08,8.83,-
Memory (GB),0.23,0.23,0.23,15.58
GPU Util (Across 1 GPUs),0.00,0.00,0.00,-
GPU Memory (GB),0.00,0.00,0.00,12.00


In [6]:
%perfmonitor_ai_review --strategy deep --cells 1:

memory_limit = 15.58
gpu_memory_limit = 12.0
ratios: {'memory': np.float64(0.025716933758471713), 'cpu': np.float64(0.05998213516129032), 'gpu_util': np.float64(0.0012903225806451613), 'gpu_memory': 1.0}
GPU idle check:
gpu_mem_avg:
7       0.000000e+00
8       0.000000e+00
9       0.000000e+00
10      0.000000e+00
11      0.000000e+00
            ...     
95      0.000000e+00
96      1.717987e+10
7868    0.000000e+00
7869    0.000000e+00
7870    0.000000e+00
Name: gpu_mem_avg, Length: 93, dtype: float64
mask_allocated:
7       False
8       False
9       False
10      False
11      False
        ...  
95      False
96       True
7868    False
7869    False
7870    False
Name: gpu_mem_avg, Length: 93, dtype: bool

gpu_util_avg:
7       0.0
8       0.0
9       0.0
10      0.0
11      0.0
       ... 
95      0.0
96      5.0
7868    0.0
7869    0.0
7870    0.0
Name: gpu_util_avg, Length: 93, dtype: float64
mask_idle:
7       True
8       True
9       True
10      True
11      True
       

## B. Memory-inefficient allocation

Code that allocates large temporaries it does not need. Expected reviewer angle: memory (`memory_limits`).


### B1 · Repeated full-length list allocation  ·  *PythonDataScienceHandbook*
**Issue:** builds a fresh N-element list every iteration just to sum it, instead of a generator or running sum.
🔗 [`01.07-Timing-and-Profiling.ipynb`](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/01.07-Timing-and-Profiling.ipynb) · cell 16

Curious how Jumper trims the memory footprint here?


In [ ]:
def sum_of_lists(N):
    total = 0
    for i in range(5):
        L = [j ^ (j >> i) for j in range(N)]
        total += sum(L)
    return total

result = sum_of_lists(5_000_000)   # original idiom, scaled up

In [ ]:
%perfmonitor_ai_review

### B2 · Full-frame zero array allocated per contour  ·  *learnopencv (OpenPose)*
**Issue:** allocates a full image-sized `np.zeros` for every detected blob, inside a per-keypoint loop.
Runs the real OpenPose COCO pose model on the shipped `group.jpg`; the setup does one forward pass, the
computation cell is the verbatim keypoint extraction loop.
🔗 [`OpenPose-Multi-Person/multi-person-openpose.py`](https://github.com/spmallick/learnopencv/blob/master/OpenPose-Multi-Person/multi-person-openpose.py#L40-L66) · lines 40-66 & 199-203


In [ ]:
%pip install -q "opencv-python<5"   # cv2 5.x removed dnn.readNetFromCaffe

import cv2
import numpy as np

# real OpenPose COCO model + sample image (shipped in demos/resources)
image1 = cv2.resize(cv2.imread("resources/group.jpg"), (2400, 1600))
frameHeight, frameWidth = image1.shape[:2]
nPoints = 18

net = cv2.dnn.readNetFromCaffe("resources/pose_deploy_linevec.prototxt",
                               "resources/pose_iter_440000.caffemodel")
net.setPreferableBackend(cv2.dnn.DNN_TARGET_CPU)
inHeight = 368
inWidth = int((inHeight / frameHeight) * frameWidth)
inpBlob = cv2.dnn.blobFromImage(image1, 1.0 / 255, (inWidth, inHeight),
                                (0, 0, 0), swapRB=False, crop=False)
net.setInput(inpBlob)
output = net.forward()

In [ ]:
def getKeypoints(probMap, threshold=0.1):
    mapSmooth = cv2.GaussianBlur(probMap, (3, 3), 0, 0)
    mapMask = np.uint8(mapSmooth > threshold)
    keypoints = []
    contours, _ = cv2.findContours(mapMask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        blobMask = np.zeros(mapMask.shape)            # full-frame allocation per contour
        blobMask = cv2.fillConvexPoly(blobMask, cnt, 1)
        maskedProbMap = mapSmooth * blobMask
        _, maxVal, _, maxLoc = cv2.minMaxLoc(maskedProbMap)
        keypoints.append(maxLoc + (probMap[maxLoc[1], maxLoc[0]],))
    return keypoints

detected_keypoints = []
for part in range(nPoints):
    probMap = cv2.resize(output[0, part, :, :], (frameWidth, frameHeight))
    detected_keypoints.append(getKeypoints(probMap, 0.1))

In [ ]:
%perfmonitor_ai_review

## C. Pandas anti-patterns

Row or group Python callbacks that hide a vectorized equivalent.

*From the **PythonDataScienceHandbook** repo.*


### C1 · Row-wise group normalization via `apply`
**Issue:** per-group Python callback mutating a copy, expressible as a vectorized `groupby().transform()`.
🔗 [`03.08-Aggregation-and-Grouping.ipynb`](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/03.08-Aggregation-and-Grouping.ipynb) · cells 47 & 58

Time to let the AI reviewer earn its keep.


In [ ]:
%pip install -q pandas

import numpy as np
import pandas as pd

# original schema (key, data1, data2); scaled to many groups so apply() is measurable
rng = np.random.RandomState(0)
N, G = 400_000, 5_000
df = pd.DataFrame({"key": rng.randint(0, G, N),
                   "data1": np.arange(N),
                   "data2": rng.randint(0, 10, N)},
                  columns=["key", "data1", "data2"])

In [ ]:
def norm_by_data2(x):
    x["data1"] /= x["data2"].sum()
    return x

result = df.groupby("key").apply(norm_by_data2)

In [ ]:
%perfmonitor_ai_review

## D. CPU to GPU parallelization

Data-parallel numeric workloads that map cleanly onto a GPU (CuPy, PyTorch-CUDA, `cv2.cuda`).
Expected reviewer angle: use `hardware_info` (`gpu_name`, `num_gpus`, `gpu_memory`) to justify offloading.


### D1 · All-pairs distance matrix plus KNN  ·  *PythonDataScienceHandbook*
**Issue:** O(N^2) broadcast distance matrix plus argsort on the CPU, a natural fit for `torch.cdist` or CuPy with `topk`.
🔗 [`02.08-Sorting.ipynb`](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/02.08-Sorting.ipynb) · cells 32, 40 & 42

Let's hand it to the reviewer and see if it reaches for the GPU.


In [ ]:
import numpy as np

# original idiom rng.random((N, 2)); scaled from 10 to a few thousand points
rng = np.random.default_rng(seed=42)
X = rng.random((5000, 2))

In [ ]:
dist_sq = np.sum((X[:, np.newaxis] - X[np.newaxis, :]) ** 2, axis=-1)
nearest = np.argsort(dist_sq, axis=1)
K = 2
nearest_partition = np.argpartition(dist_sq, K + 1, axis=1)

In [ ]:
%perfmonitor_ai_review

### D2 · Element-wise math over a 2D grid  ·  *PythonDataScienceHandbook*
**Issue:** transcendental element-wise ops on a broadcast grid, a pure CuPy drop-in (`import cupy as np`).
🔗 [`02.05-Computation-on-arrays-broadcasting.ipynb`](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/02.05-Computation-on-arrays-broadcasting.ipynb) · cell 49


In [ ]:
import numpy as np

# original idiom linspace(0, 5, n); scaled from 50 to a large grid
x = np.linspace(0, 5, 7000)
y = np.linspace(0, 5, 7000)[:, np.newaxis]

In [ ]:
z = np.sin(x) ** 10 + np.cos(10 + y * x) * np.cos(x)

In [ ]:
%perfmonitor_ai_review

*The remaining examples in this section are from the **learnopencv** repo (spmallick).*


### D3 · Pairwise squared and cosine distance  ·  *DeepSORT*
**Issue:** per-frame N x L cost matrices built from GEMM (`np.dot`) on the CPU, offloadable to `torch.mm` on-device.
🔗 [`ALPR/DeepSORT-tracker/.../nn_matching.py`](https://github.com/spmallick/learnopencv/blob/master/ALPR/DeepSORT-tracker/deep_sort/deep_sort/nn_matching.py#L24-L53) · lines 24-53

Wonder what the Jumper reviewer proposes for a per-frame hot path like this.


In [ ]:
import numpy as np

# nn_matching.py has no fixed inputs; features are 128-d appearance descriptors (DeepSORT default)
rs = np.random.RandomState(0)
a = rs.rand(7000, 128)   # track features
b = rs.rand(7000, 128)   # detection features

In [ ]:
def _pdist(a, b):
    a, b = np.asarray(a), np.asarray(b)
    a2, b2 = np.square(a).sum(axis=1), np.square(b).sum(axis=1)
    r2 = -2. * np.dot(a, b.T) + a2[:, None] + b2[None, :]
    return np.clip(r2, 0., float(np.inf))

def _cosine_distance(a, b):
    a = np.asarray(a) / np.linalg.norm(a, axis=1, keepdims=True)
    b = np.asarray(b) / np.linalg.norm(b, axis=1, keepdims=True)
    return 1. - np.dot(a, b.T)

sq_cost = _pdist(a, b)
cos_cost = _cosine_distance(a, b)

In [ ]:
%perfmonitor_ai_review

### D4 · K-means anchor IoU loop  ·  *YOLO ALPR*
**Issue:** per-box Python loop over the whole dataset, repeated every k-means iteration, instead of a batched IoU tensor.
🔗 [`ALPR/License-plate-detection/darknet/scripts/gen_anchors.py`](https://github.com/spmallick/learnopencv/blob/master/ALPR/License-plate-detection/darknet/scripts/gen_anchors.py#L30-L44) · lines 30-44


In [ ]:
import numpy as np

# original reads (w, h) box dims from a VOC annotation filelist; synthesized here at dataset scale
rs = np.random.RandomState(0)
X = rs.rand(150_000, 2)       # (w, h) of 150k boxes
centroids = rs.rand(9, 2)     # 9 anchor centroids

In [ ]:
def IOU(x, centroids):
    similarities = []
    w, h = x
    for c_w, c_h in centroids:
        if c_w >= w and c_h >= h:
            similarity = w * h / (c_w * c_h)
        elif c_w >= w and c_h <= h:
            similarity = w * c_h / (w * h + (c_w - w) * c_h)
        elif c_w <= w and c_h >= h:
            similarity = c_w * h / (w * h + c_w * (c_h - h))
        else:
            similarity = (c_w * c_h) / (w * h)
        similarities.append(similarity)
    return np.array(similarities)

def avg_IOU(X, centroids):
    n, d = X.shape
    total = 0.
    for i in range(X.shape[0]):
        total += max(IOU(X[i], centroids))
    return total / n

result = avg_IOU(X, centroids)

In [ ]:
%perfmonitor_ai_review

### D5 · Embedding extraction with `batch_size=1`  ·  *ArcFace*
**Issue:** GPU-capable backbone fed one image per forward pass; raising the batch size would saturate the device.
Runs the real IR-50 (ms1m) ArcFace backbone over the shipped face crops, exactly as in `embeddings.py`.
🔗 [`Face-Recognition-with-ArcFace/embeddings.py`](https://github.com/spmallick/learnopencv/blob/master/Face-Recognition-with-ArcFace/embeddings.py#L42-L59) · lines 42 & 54-59

Let's put the reviewer on this and watch it batch things up.


In [ ]:
%pip install -q torch torchvision   # CPU build is sufficient

import sys
import numpy as np
import torch
import torch.nn.functional as F
import torch.utils.data as data
import torchvision.transforms as transforms
from torchvision import datasets

# demos/resources ships backbone.py, the IR-50 weights, and the face crops
sys.path.insert(0, "resources")
from arcface_backbone import Backbone

input_size = [112, 112]
transform = transforms.Compose([
    transforms.Resize([int(128 * input_size[0] / 112)] * 2),
    transforms.CenterCrop(input_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])
dataset = datasets.ImageFolder("resources/arcface_faces", transform)
loader = data.DataLoader(dataset, batch_size=1, shuffle=False, pin_memory=True, num_workers=0)

backbone = Backbone(input_size)
backbone.load_state_dict(torch.load("resources/backbone_ir50_ms1m_epoch120.pth",
                                    map_location=torch.device("cpu")))
backbone.eval()

In [ ]:
embeddings = np.zeros([len(loader.dataset), 512])
with torch.no_grad():
    for idx, (image, _) in enumerate(loader):        # batch_size == 1
        embeddings[idx, :] = F.normalize(backbone(image)).cpu()

In [ ]:
%perfmonitor_ai_review

### D6 · Per-channel normalization loop  ·  *FBA Matting*
**Issue:** channel-by-channel host-side normalization inside a GPU pipeline, better vectorized over the channel axis or done on-device.
🔗 [`FBAMatting/networks/transforms.py`](https://github.com/spmallick/learnopencv/blob/master/FBAMatting/networks/transforms.py#L30-L43) · lines 30-43


In [ ]:
%pip install -q pillow

import numpy as np
from PIL import Image

# real sample image shipped with the FBA Matting repo, upscaled to full-frame resolution
_p = "resources/boy.png"   # shipped in demos/resources
img = np.asarray(Image.open(_p).convert("RGB").resize((5500, 5500)), dtype="float32") / 255.0
group_norm_mean = [0.485, 0.456, 0.406]
group_norm_std = [0.229, 0.224, 0.225]

In [ ]:
for i in range(3):
    img[..., i] = (img[..., i] - group_norm_mean[i]) / group_norm_std[i]

In [ ]:
%perfmonitor_ai_review